In [1]:
# ============================================================
#  10 OD & Ground Friction
#  输入: 04 路由图 + 05 barrier unions
#  输出: OD 分析表 + 路径集合
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
from shapely.ops import unary_union
from math import radians, sin, cos, sqrt, atan2
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

GRAPH_PATH = Path("../04 Transport/data_out/sz_drive.graphml")
BARRIER_UNIONS_PATH = Path("../05 Barrier Layers/data_out/sz_barrier_unions.gpkg")
EDGES_PATH = Path("../04 Transport/data_out/sz_drive_edges.gpkg")
BOUNDARY_PATH = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")

def haversine_m(lon1, lat1, lon2, lat2):
    R = 6_371_000
    la1, la2 = radians(lat1), radians(lat2)
    a = sin((la2 - la1) / 2) ** 2 + cos(la1) * cos(la2) * sin(radians(lon2 - lon1) / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# ── 加载路由图 ──
print("Loading graph ...")
G = ox.load_graphml(GRAPH_PATH)
G = ox.routing.add_edge_speeds(G, fallback=30)
G = ox.routing.add_edge_travel_times(G)
print(f"  Graph: {len(G.nodes):,} nodes, {len(G.edges):,} edges")

# ── 加载 barrier unions ──
print("Loading barrier unions ...")
barrier_unions = gpd.read_file(BARRIER_UNIONS_PATH)
barriers = {}
for _, row in barrier_unions.iterrows():
    barriers[row["barrier_type"]] = row.geometry
print(f"  Barrier types: {list(barriers.keys())}")

# ── 加载深圳边界 ──
shenzhen = gpd.read_file(BOUNDARY_PATH).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union

print("Ready.")

/Users/shirly/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading graph ...
  Graph: 409,106 nodes, 661,793 edges
Loading barrier unions ...
  Barrier types: ['water', 'waterway', 'railway', 'highway_major', 'bridges', 'tunnels']
Ready.


In [ ]:
# ============================================================
#  1. 生成 OD 矩阵
#  在深圳范围内按网格采样, 过滤掉水体/边界外的点
#  然后生成所有 OD 对 (可控制采样密度)
# ============================================================
from shapely.prepared import prep as shapely_prep
from itertools import combinations

GRID_SPACING_DEG = 0.02  # ~2.2 km 间距, 可调小 (0.01 ~ 1.1 km) 获得更密的网格

minx, miny, maxx, maxy = shenzhen_geom.bounds
sz_prep = shapely_prep(shenzhen_geom)

# 水体 union 用于排除落在水面上的采样点
water_geom = barriers.get("water")

# 生成候选网格点
xs = np.arange(minx, maxx, GRID_SPACING_DEG)
ys = np.arange(miny, maxy, GRID_SPACING_DEG)
candidates = []
for x in xs:
    for y in ys:
        pt = Point(x, y)
        if not sz_prep.contains(pt):
            continue
        if water_geom and water_geom.contains(pt):
            continue
        candidates.append(pt)

print(f"Grid spacing: {GRID_SPACING_DEG}° (~{GRID_SPACING_DEG * 111:.1f} km)")
print(f"Candidate OD points: {len(candidates)}")

# 为每个点找最近的图节点
od_nodes = []
for pt in candidates:
    nid = ox.nearest_nodes(G, pt.x, pt.y)
    od_nodes.append({"node_id": nid, "lon": pt.x, "lat": pt.y, "geometry": pt})

od_points = gpd.GeoDataFrame(od_nodes, crs=4326).drop_duplicates(subset="node_id")
print(f"Unique OD nodes (after dedup): {len(od_points)}")

# 生成 OD 对 (可控制采样数量)
import random

MAX_OD_PAIRS = None  # 全量跑: 所有 OD 对

all_pairs = list(combinations(range(len(od_points)), 2))
if MAX_OD_PAIRS and len(all_pairs) > MAX_OD_PAIRS:
    random.seed(42)
    od_pairs = random.sample(all_pairs, MAX_OD_PAIRS)
    print(f"OD pairs: sampled {len(od_pairs):,} from {len(all_pairs):,} total")
else:
    od_pairs = all_pairs
    print(f"OD pairs to compute: {len(od_pairs):,}")

Grid spacing: 0.02° (~2.2 km)
Candidate OD points: 412


In [ ]:
# ============================================================
#  2. 批量路由计算 (free-flow + peak-hour)
#  对每对 OD 同时计算两种通行时间, 得到 congestion amplifier
# ============================================================
from tqdm.auto import tqdm
import copy

# ── 构建高峰图: 对不同道路类型施加速度折扣 ──
PEAK_DISCOUNT = {
    "motorway": 0.7,
    "motorway_link": 0.7,
    "trunk": 0.5,
    "trunk_link": 0.5,
    "primary": 0.4,
    "primary_link": 0.4,
    "secondary": 0.5,
    "secondary_link": 0.5,
    "tertiary": 0.6,
    "tertiary_link": 0.6,
    "residential": 0.8,
    "living_street": 0.9,
    "unclassified": 0.7,
    "service": 0.9,
}

print("Building peak-hour graph ...")
G_peak = copy.deepcopy(G)
for u, v, k, data in G_peak.edges(data=True, keys=True):
    hw = data.get("highway", "")
    if isinstance(hw, list):
        hw = hw[0]
    discount = PEAK_DISCOUNT.get(hw, 0.7)
    if "speed_kph" in data and "length" in data:
        peak_speed = data["speed_kph"] * discount
        data["peak_speed_kph"] = peak_speed
        data["peak_travel_time"] = data["length"] / (peak_speed / 3.6)
    else:
        data["peak_travel_time"] = data.get("travel_time", 0) / discount
print("  Peak graph ready.")

# ── 批量路由 ──
pts = od_points.reset_index(drop=True)
results = []

print(f"\nComputing {len(od_pairs):,} routes (free-flow + peak) ...")
for i, j in tqdm(od_pairs, desc="Routing"):
    o_row = pts.iloc[i]
    d_row = pts.iloc[j]
    o_node = o_row["node_id"]
    d_node = d_row["node_id"]

    try:
        path = ox.shortest_path(G, o_node, d_node, weight="travel_time")
    except Exception:
        path = None
    if path is None or len(path) < 2:
        continue

    try:
        route_gdf = ox.routing.route_to_gdf(G, path)
    except Exception:
        continue

    net_m = route_gdf["length"].sum()
    tt_free = route_gdf["travel_time"].sum()
    fly_m = haversine_m(o_row["lon"], o_row["lat"], d_row["lon"], d_row["lat"])
    detour = net_m / fly_m if fly_m > 0 else np.nan

    # 高峰通行时间: 沿同一路径, 用 peak_travel_time
    tt_peak = 0
    for pi in range(len(path) - 1):
        edge_data = G_peak[path[pi]][path[pi + 1]]
        best = min(edge_data.values(), key=lambda d: d.get("peak_travel_time", 9999))
        tt_peak += best.get("peak_travel_time", 0)

    congestion_amp = tt_peak / tt_free if tt_free > 0 else 1.0

    od_line = LineString([(o_row["lon"], o_row["lat"]), (d_row["lon"], d_row["lat"])])
    route_line = LineString([(G.nodes[n]["x"], G.nodes[n]["y"]) for n in path])

    n_bridges = int(route_gdf["bridge"].eq("yes").sum()) if "bridge" in route_gdf.columns else 0
    n_tunnels = int(route_gdf["tunnel"].eq("yes").sum()) if "tunnel" in route_gdf.columns else 0

    results.append({
        "o_idx": i, "d_idx": j,
        "o_node": o_node, "d_node": d_node,
        "o_lon": o_row["lon"], "o_lat": o_row["lat"],
        "d_lon": d_row["lon"], "d_lat": d_row["lat"],
        "net_m": net_m, "fly_m": fly_m,
        "tt_free_s": tt_free, "tt_peak_s": tt_peak,
        "congestion_amplifier": congestion_amp,
        "detour_ratio": detour,
        "n_bridges_used": n_bridges, "n_tunnels_used": n_tunnels,
        "od_line": od_line, "route_line": route_line,
    })

print(f"\nCompleted: {len(results):,} valid routes out of {len(od_pairs):,} pairs")

Building peak-hour graph ...
  Peak graph ready.

Computing 2,000 routes (free-flow + peak) ...


TqdmKeyError: "Unknown argument(s): {'miniinterval': 2}"

In [ ]:
# ============================================================
#  3. Barrier 交叉分析
#  对每对 OD 的直线, 判定是否穿越各类 barrier + 穿越次数
# ============================================================
from shapely.prepared import prep as shapely_prep

# 预构建 prepared geometry 加速查询
barrier_prep = {}
for btype, geom in barriers.items():
    if geom is not None:
        barrier_prep[btype] = shapely_prep(geom)

def count_crossings(line, barrier_geom):
    """计算 line 与 barrier 的交叉次数 (交叉点/线段数)."""
    if barrier_geom is None:
        return 0
    try:
        inter = line.intersection(barrier_geom)
        if inter.is_empty:
            return 0
        if inter.geom_type == "Point":
            return 1
        if inter.geom_type == "MultiPoint":
            return len(inter.geoms)
        if inter.geom_type in ("LineString", "MultiLineString", "GeometryCollection"):
            return len(list(inter.geoms)) if hasattr(inter, "geoms") else 1
    except Exception:
        return 0
    return 1

print(f"Analyzing barrier crossings for {len(results):,} OD pairs ...")

for r in tqdm(results, desc="Barrier analysis"):
    od_line = r["od_line"]

    for btype in ["water", "waterway", "railway", "highway_major"]:
        prep = barrier_prep.get(btype)
        geom = barriers.get(btype)

        r[f"crosses_{btype}"] = bool(prep.intersects(od_line)) if prep else False
        r[f"n_{btype}_crossings"] = count_crossings(od_line, geom)

    r["n_barrier_total"] = sum(r.get(f"n_{bt}_crossings", 0)
                                for bt in ["water", "waterway", "railway", "highway_major"])
    r["bridge_dependent"] = r["n_bridges_used"] > 0
    r["tunnel_dependent"] = r["n_tunnels_used"] > 0

print("Done.")

In [ ]:
# ============================================================
#  3b. Overload Correction Layer
#  读取 09 人口/活动网格, 为每对 OD 附加起终点的活动强度
#  作为 ground_friction 的修正项: 高活动区域 → 拥堵更严重
# ============================================================

POP_GRID_PATH = Path("../09 Population/data_out/sz_population_grid.gpkg")

if POP_GRID_PATH.exists():
    print("Loading 09 population grid for overload correction ...")
    pop_grid = gpd.read_file(POP_GRID_PATH)

    from scipy.spatial import cKDTree

    grid_centroids = np.array(list(zip(
        pop_grid.geometry.centroid.x, pop_grid.geometry.centroid.y
    )))
    tree = cKDTree(grid_centroids)

    def get_grid_value(lon, lat, col):
        _, idx = tree.query([lon, lat])
        return pop_grid.iloc[idx][col] if col in pop_grid.columns else 0

    for r in tqdm(results, desc="Overload correction"):
        o_intensity = get_grid_value(r["o_lon"], r["o_lat"], "intensity_index")
        d_intensity = get_grid_value(r["d_lon"], r["d_lat"], "intensity_index")
        r["origin_intensity"] = o_intensity
        r["dest_intensity"] = d_intensity
        r["overload_factor"] = (o_intensity + d_intensity) / 2

    del pop_grid, tree
    print(f"  Done. Mean overload_factor: {np.mean([r['overload_factor'] for r in results]):.3f}")
else:
    print("09 population grid not found. Setting overload_factor = 0.")
    for r in results:
        r["origin_intensity"] = 0
        r["dest_intensity"] = 0
        r["overload_factor"] = 0

In [ ]:
# ============================================================
#  4. 合成 Ground Friction 指标 + 保存
#  包含: detour + barrier + bridge/tunnel + congestion amplifier
# ============================================================

df = pd.DataFrame(results)

# ── 归一化各分量到 [0, 1] ──
def norm_col(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0

df["detour_norm"] = norm_col(df["detour_ratio"])
df["barrier_norm"] = norm_col(df["n_barrier_total"].astype(float))
df["bridge_score"] = df["bridge_dependent"].astype(float)
df["tunnel_score"] = df["tunnel_dependent"].astype(float)
df["congestion_norm"] = norm_col(df["congestion_amplifier"])
df["overload_norm"] = norm_col(df["overload_factor"]) if "overload_factor" in df.columns else 0

# ── 加权合成 ──
W_DETOUR     = 0.25
W_BARRIER    = 0.20
W_BRIDGE     = 0.08
W_TUNNEL     = 0.07
W_CONGESTION = 0.20
W_OVERLOAD   = 0.20

df["ground_friction"] = (
    W_DETOUR     * df["detour_norm"]
  + W_BARRIER    * df["barrier_norm"]
  + W_BRIDGE     * df["bridge_score"]
  + W_TUNNEL     * df["tunnel_score"]
  + W_CONGESTION * df["congestion_norm"]
  + W_OVERLOAD   * df["overload_norm"]
)

# ── 保存 OD 分析表 (几何=OD 直线) ──
od_gdf = gpd.GeoDataFrame(df, geometry="od_line", crs=4326)
od_gdf = od_gdf.drop(columns=["route_line"])
od_gdf.to_file(OUT / "sz_od_analysis.gpkg", driver="GPKG")
print(f"OD analysis -> data_out/sz_od_analysis.gpkg  ({len(od_gdf):,} rows)")

# ── 保存路径集合 (几何=实际路径) ──
routes_gdf = gpd.GeoDataFrame(
    df[["o_idx", "d_idx", "detour_ratio", "congestion_amplifier", "ground_friction", "route_line"]],
    geometry="route_line", crs=4326,
)
routes_gdf.to_file(OUT / "sz_routes.gpkg", driver="GPKG")
print(f"Routes     -> data_out/sz_routes.gpkg        ({len(routes_gdf):,} rows)")

# ── 统计摘要 ──
print("\n=== Summary ===")
print(f"Valid OD pairs:      {len(df):,}")
print(f"Detour ratio:        mean={df['detour_ratio'].mean():.3f}  median={df['detour_ratio'].median():.3f}  max={df['detour_ratio'].max():.3f}")
print(f"Congestion amp:      mean={df['congestion_amplifier'].mean():.2f}  median={df['congestion_amplifier'].median():.2f}  max={df['congestion_amplifier'].max():.2f}")
print(f"Ground friction:     mean={df['ground_friction'].mean():.3f}  median={df['ground_friction'].median():.3f}  max={df['ground_friction'].max():.3f}")
print(f"\nBarrier crossing rates:")
for bt in ["water", "waterway", "railway", "highway_major"]:
    rate = df[f"crosses_{bt}"].mean() * 100
    avg_n = df[f"n_{bt}_crossings"].mean()
    print(f"  {bt:20s}  {rate:5.1f}% of ODs cross  |  avg crossings: {avg_n:.2f}")
print(f"  bridge_dependent:   {df['bridge_dependent'].mean()*100:.1f}%")
print(f"  tunnel_dependent:   {df['tunnel_dependent'].mean()*100:.1f}%")